# Data Cleaning

This notebook rebuilds the original R data-cleaning workflow in Python. The goal is to transform the raw MTurk / survey data into a cleaner dataset that can be used for imputation, exploratory analysis, and clustering.

Main cleaning decisions:

- Treat `NA` and `#N/A` as missing values.
- Remove platform identifiers, survey codes, IP/location fields, high-cardinality timestamps, and constant or near-identifier variables.
- Convert `CreationTime` into a categorical `batch` variable.
- Keep work-time variables as potential data quality indicators.
- Convert `Prefer not to state` in age and income into missing values.
- Simplify `Other / Prefer not to state` in gender to `Other`.
- Save the cleaned dataset for downstream imputation and clustering.


## 1. Import Libraries and Load Data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Data_Cleaning" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "Paper and Materials" / "Raw Data.csv"
OUTPUT_PATH = PROJECT_ROOT / "Data_Cleaning" / "clean_data_python.csv"

raw = pd.read_csv(RAW_DATA_PATH, na_values=["NA", "#N/A"])
raw.shape

## 2. Missing Value Overview

First, inspect how many values are missing in each column. This helps identify variables affected by follow-up survey non-response and variables that may require imputation later.

In [ ]:
missing_summary = (
    raw.isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "variable"})
)
missing_summary["pct_missing"] = missing_summary["n_missing"] / len(raw) * 100
missing_summary.sort_values("n_missing", ascending=False)

## 3. Inspect Low-Value or Non-Analytic Columns

Several fields are useful for data collection but not useful as clustering features. Examples include assignment IDs, worker IDs, IP addresses, timestamps, and survey codes.

In [ ]:
columns_to_check = [
    "AssignmentId",
    "HITTypeId",
    "Reward",
    "MaxAssignments",
    "CreationTime",
    "SubmitTime",
    "Answer.surveycode",
]

unique_counts = []
for col in columns_to_check:
    if col in raw.columns:
        unique_counts.append(
            {
                "column": col,
                "n_unique": raw[col].nunique(dropna=True),
                "example_values": raw[col].dropna().astype(str).unique()[:5].tolist(),
            }
        )

pd.DataFrame(unique_counts)

## 4. Convert Creation Time to Batch

`CreationTime` has a small number of distinct values, which appear to represent questionnaire batches. Instead of keeping the full timestamp, convert it into a categorical `batch` variable.

In [ ]:
data = raw.copy()

if "CreationTime" in data.columns:
    batch_labels = {value: chr(ord("a") + idx) for idx, value in enumerate(sorted(data["CreationTime"].dropna().unique()))}
    data["batch"] = data["CreationTime"].map(batch_labels).astype("category")
    data = data.drop(columns=["CreationTime"])

data["batch"].value_counts(dropna=False).sort_index() if "batch" in data.columns else None

## 5. Remove Non-Analytic Columns

The following columns are removed because they are identifiers, privacy-related fields, constant fields, high-cardinality timestamps, or collection metadata rather than behavioral features.

In [ ]:
columns_to_drop = [
    "...27",
    "...28",
    "WorkerId",
    "IPAddress",
    "LocationLatitude",
    "LocationLongitude",
    "AssignmentId",
    "HITTypeId",
    "Reward",
    "AcceptTime...7",
    "AcceptTime...16",
    "MaxAssignments",
    "SubmitTime",
    "Answer.surveycode",
]

clean_data = data.drop(columns=[col for col in columns_to_drop if col in data.columns])
clean_data.shape

## 6. Data Quality Check: Work Time

Work time is not removed because it may be useful as a data quality indicator. The original R analysis found one very fast response, but not enough to justify broad sample removal at this stage.

In [ ]:
work_time_col = "WorkTimeInSeconds...9"

if work_time_col in data.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(data=data, x=work_time_col, bins=30, ax=ax)
    ax.set_title("Distribution of Work Time")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Count")
    plt.show()

    print("Minimum work time:", data[work_time_col].min(skipna=True))
    print("Responses under 10 seconds:", (data[work_time_col] < 10).sum())

## 7. Inspect Core Variable Distributions

In [ ]:
if "multiplier" in clean_data.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(data=clean_data, x="multiplier", bins=30, ax=ax)
    ax.set_title("Distribution of Multiplier")
    plt.show()

for col in ["age", "gender", "income"]:
    if col in clean_data.columns:
        display(clean_data[col].value_counts(dropna=False).rename_axis(col).reset_index(name="count"))

## 8. Exploratory Checks: Donation Amount vs. Attitude Variables

These plots are exploratory rather than strict cleaning steps. They help show whether political and religious variables may be useful for later cluster interpretation.

In [ ]:
religious_col = "Q8_1 On a scale of 0 to 100, how would you describe your religious orientation?"
political_col = "Q3_1 On a scale of 0 to 100, how would you describe your political views?"
attendance_col = "Q8 Aside from weddings and funerals, how often do you attend religious services?"

for x_col, title in [
    (religious_col, "Amount vs. Religious Orientation"),
    (political_col, "Amount vs. Political Views"),
    (attendance_col, "Amount vs. Religious Service Attendance"),
]:
    if x_col in clean_data.columns and "amount" in clean_data.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.scatterplot(data=clean_data, x=x_col, y="amount", alpha=0.5, ax=ax)
        ax.set_title(title)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

## 9. Missingness After Dropping Columns

In [ ]:
clean_missing_summary = (
    clean_data.isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "variable"})
)
clean_missing_summary["pct_missing"] = clean_missing_summary["n_missing"] / len(clean_data) * 100
clean_missing_summary.sort_values("n_missing", ascending=False)

## 10. Standardize Special Response Categories

`Prefer not to state` is not a true age or income level, so it is converted to missing. For gender, the combined category `Other / Prefer not to state` is simplified to `Other`.

In [ ]:
prefer_not_counts = []
for col in clean_data.columns:
    count = (clean_data[col] == "Prefer not to state").sum() if clean_data[col].dtype == "object" else 0
    if count > 0:
        prefer_not_counts.append({"column": col, "count": int(count)})

pd.DataFrame(prefer_not_counts)

In [ ]:
for col in ["age", "income"]:
    if col in clean_data.columns:
        clean_data[col] = clean_data[col].replace("Prefer not to state", pd.NA)

if "gender" in clean_data.columns:
    clean_data["gender"] = clean_data["gender"].replace("Other / Prefer not to state", "Other")

clean_data[[col for col in ["age", "gender", "income"] if col in clean_data.columns]].head()

## 11. Save Cleaned Data

This output is the Python equivalent of the original `clean_data.csv` and can be used as input for the imputation notebook.

In [ ]:
clean_data.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned data to: {OUTPUT_PATH}")
clean_data.head()